# Module 5.1: Retrieval Is Policy, Not Just Similarity

Sarah Chen's memory store now holds dozens of preferences. Each one carries a lifecycle state, a confidence score, a validity window and provenance. Sarah asks:

> **"Book me a hotel for my Chicago trip next week. What do you know about how I like to stay?"**

Several memories are about hotels. Some are current and trusted. One was replaced months ago. One is an unconfirmed inference. One was retired by the retention sweep. A vector search scores them all as *related*.

**Which memories should actually reach the agent's context?**

In [ ]:
%pip install -q -r ../requirements.txt

In [ ]:
import os
import sys

import certifi
import sniffio

sys.path.insert(0, "..")
sniffio.current_async_library_cvar.set("asyncio")
os.environ["SSL_CERT_FILE"] = certifi.where()

from dotenv import load_dotenv
from azure.cosmos.aio import CosmosClient
from azure.identity.aio import (
    AzureCliCredential as AsyncCliCredential,
    get_bearer_token_provider as async_get_bearer_token_provider,
)
from openai import AsyncAzureOpenAI
from agent_framework import Agent, AgentSession, tool
from shared.memory_tools import TRUST_STATES, make_retrieval_tool, presentation, retrieve
from shared.semantic_store import SemanticMemoryStore, create_container
from shared.travel_agent import SYSTEM_PROMPT, create_client

load_dotenv("../.env", override=True)
client, credential = create_client("../.env")
print("Foundry client ready")

In [ ]:
embed_token = async_get_bearer_token_provider(
    AsyncCliCredential(), "https://cognitiveservices.azure.com/.default"
)
embed_deployment = os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-ada-002")
embed_client = AsyncAzureOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    azure_ad_token_provider=embed_token,
    api_version="2024-02-01",
)

async def embed(text: str) -> list[float]:
    r = await embed_client.embeddings.create(input=[text], model=embed_deployment)
    return r.data[0].embedding

SCOPE = "nb05_retrieval"
cosmos = CosmosClient(os.environ["COSMOS_ENDPOINT"], credential=AsyncCliCredential())
container = await create_container(cosmos)
store = SemanticMemoryStore(container, user_id="E001", embed_fn=embed, scope_tag=SCOPE)
print(f"Cleared {await store.reset()} documents from a previous run")

## Seeding a Realistic Memory Store

The seed mirrors what Modules 3 and 4 leave behind: trusted facts, provisional beliefs, a quarantined inference, a superseded version and a retired memory. Several are deliberately similar to a hotel question. The `scope_tag` keeps this demo's documents separate from the earlier notebooks' data in Sarah's partition.

In [ ]:
# (category, preference, confidence, source_type, state)
SEED = [
    ("hotel_chain", "Prefers to stay at Hilton hotels", 0.90, "user_assertion", "trusted"),
    ("hotel_chain", "Prefers to stay at Marriott hotels", 0.95, "user_assertion", "trusted"),  # supersedes Hilton
    ("lodging_style", "Prefers Airbnb apartments over hotels", 0.90, "llm_inference", "candidate"),
    ("budget_hotel", "Prefers Holiday Inn Express hotels", 0.85, "user_assertion", "deprecated"),
    ("solo_lodging", "Stays at hostels when travelling solo", 0.60, "llm_inference", "deprecated"),
    ("hotel_loyalty", "Is a Marriott Bonvoy Gold Elite member", 0.95, "user_assertion", "trusted"),
    ("hotel_room", "Prefers a quiet room on a high floor", 0.70, "user_assertion", "trusted"),
    ("hotel_location", "Wants hotels within walking distance of the client office", 0.60, "user_assertion", "provisional"),
    ("hotel_breakfast", "Wants breakfast included in the hotel rate", 0.75, "user_assertion", "trusted"),
    ("hotel_checkout", "Usually asks for late checkout", 0.55, "user_assertion", "provisional"),
    ("hotel_gym", "Needs a hotel gym for morning workouts", 0.40, "llm_inference", "provisional"),
    ("dietary", "Likes very spicy food", 0.50, "llm_inference", "candidate"),
    ("seating", "Prefers aisle seats on flights", 0.70, "user_assertion", "provisional"),
    ("airline", "Flies Air France for long-haul trips", 0.90, "user_assertion", "trusted"),
    ("home_city", "Lives in Paris", 0.95, "user_assertion", "trusted"),
    ("meal", "Orders vegetarian meals on flights", 0.90, "user_assertion", "trusted"),
    ("flight_time", "Prefers morning departures before 10am", 0.85, "user_assertion", "trusted"),
    ("car_rental", "Rents compact cars from Hertz", 0.80, "user_assertion", "trusted"),
    ("lounge", "Uses airport lounges on layovers over two hours", 0.70, "user_assertion", "trusted"),
    ("connections", "Avoids layovers shorter than 90 minutes", 0.85, "user_assertion", "trusted"),
    ("language", "Speaks English and French", 0.95, "user_assertion", "trusted"),
    ("expenses", "Submits expenses in euros", 0.90, "user_assertion", "trusted"),
    ("connectivity", "Needs reliable Wi-Fi for a work laptop", 0.80, "user_assertion", "trusted"),
]

for category, text, confidence, source_type, state in SEED:
    await store.add_preference(category, text, confidence, source_type, state=state)

docs = await store.snapshot(include_deprecated=True)
superseded = [d["preference"] for d in docs if d["valid_to"]]
assert superseded == ["Prefers to stay at Hilton hotels"], superseded
print(f"Seeded {len(docs)} memories; superseded: {superseded}")

## The Problem: Naive Vector Search

The simplest retrieval embeds the question and returns the nearest documents. The Cosmos partition key already restricts results to Sarah's `/user_id`, but no other lifecycle rule is applied.

In [ ]:
QUERY = "hotel preferences for a business trip to Chicago"

def show(rows, extra=()):
    for i, r in enumerate(rows, 1):
        current = "current" if r.get("valid_to") is None else "SUPERSEDED"
        cols = [f"{r['score']:.3f}", f"{r['state']:<11}", f"{current:<10}", f"conf={r['confidence']:.2f}"]
        cols += [f"{k}={r[k]:.3f}" if isinstance(r[k], float) else f"{k}={r[k]}" for k in extra]
        print(f"{i:>2}. " + "  ".join(cols) + f"  {r['preference']}")

naive = await store.search(QUERY, top_k=10, current_only=False, states=None, track_access=False)
show(naive)

In [ ]:
@tool
async def recall_preferences_naive(query: str) -> str:
    """Recall the user's stored preferences related to the query."""
    rows = await store.search(query, top_k=10, current_only=False, states=None, track_access=False)
    return "\n".join(f"[{r['category']}] {r['preference']}" for r in rows)

QUESTION = (
    "Book me a hotel for my Chicago trip next week. "
    "What do you know about how I like to stay?"
)

naive_agent = Agent(
    client=client,
    name="NaiveRetrievalAssistant",
    instructions=SYSTEM_PROMPT + "\n\nBefore personalizing, call recall_preferences_naive and use what it returns.",
    tools=[recall_preferences_naive],
)
naive_answer = await naive_agent.run(QUESTION, session=AgentSession())
print("ASSISTANT:", naive_answer.text)

## What Went Wrong

Every result is *similar* to the question. Several are not *eligible* to influence the answer:

| Result | Why it is related | Why it must not reach context |
|---|---|---|
| Hilton | Same category as the current hotel chain | Superseded by Marriott (`valid_to` is set) |
| Airbnb | Lodging preference, confidence 0.90 | `candidate` — an unconfirmed inference, quarantined |
| Holiday Inn Express, hostels | Lodging preferences | `deprecated` — retired by the retention sweep |
| Low-confidence gym inference | Hotel amenity | Eligible, but should be asked about, not asserted |
| Ten results | All loosely related | Spends context on records the agent cannot use |

Lifecycle work from Modules 3 and 4 is still in the store, but naive retrieval ignores it. The agent receives contradictory hotel chains with no way to tell which one is current.

## The Solution: A Lifecycle-Aware Retrieval Pipeline

```mermaid
flowchart LR
    Q[Query] --> V[DiskANN search]
    V --> U[User partition + scope]
    U --> C[Current versions only]
    C --> T[Trust-state gate]
    T --> R[Rank: relevance + confidence]
    R --> B[top_k budget]
    B --> A[Agent context]
```

Each stage is a filter or ordering over the same `SemanticMemoryStore.search()` call. We add them one at a time.

### Step 1: Current versions only

`current_only=True` adds `IS_NULL(c.valid_to)` to the query, so superseded SCD-2 versions never come back.

In [ ]:
step1 = await store.search(QUERY, top_k=10, current_only=True, track_access=False)
show(step1)
assert all(r["valid_to"] is None for r in step1)

### Step 2: Trust-state gate

`states=TRUST_STATES` keeps only `trusted` and `provisional`. A confidence of 0.90 does not let the Airbnb candidate through; state controls visibility first.

In [ ]:
print("TRUST_STATES =", TRUST_STATES)
step2 = await store.search(QUERY, top_k=20, current_only=True, states=TRUST_STATES, track_access=False)
show(step2)
assert {r["state"] for r in step2} <= set(TRUST_STATES)

### Step 3: Rank by relevance and confidence

Similarity alone ranks a weak inference above a confirmed fact if its wording happens to be closer. We blend the two. Embedding similarities cluster in a narrow band (here roughly 0.70 to 0.85), so similarity is first rescaled to $[0, 1]$ within the candidate pool; otherwise even a small confidence weight would override relevance:

$$\text{relevance} = \frac{s - s_{min}}{s_{max} - s_{min}}, \qquad \text{rank} = (1 - w)\cdot\text{relevance} + w\cdot\text{confidence}, \quad w = 0.3$$

Relevance still dominates; confidence breaks near-ties.

In [ ]:
W = 0.3
lo, hi = min(r["score"] for r in step2), max(r["score"] for r in step2)
for r in step2:
    r["rank"] = (1 - W) * (r["score"] - lo) / (hi - lo) + W * r["confidence"]
step3 = sorted(step2, key=lambda r: r["rank"], reverse=True)
show(step3, extra=["rank"])

### Step 4: Bound the context

A fixed `top_k` caps how many memories the agent sees. The low-ranked, low-confidence items fall off without any change to storage.

In [ ]:
TOP_K = 5
step4 = step3[:TOP_K]
for r in step4:
    r["presentation"] = presentation(r["state"], r["confidence"])
show(step4, extra=["rank", "presentation"])
print(f"\nContext: {len(naive)} naive results -> {len(step4)} eligible, ranked memories")

## Packaging the Pipeline

The four steps are packaged as `shared.memory_tools.retrieve(store, query, top_k=5)`. It also marks **only the returned memories** as accessed, so the Module 3 retention scorer sees real usage signals rather than every candidate the search touched.

In [ ]:
def access_counts():
    return {d["id"]: d.get("access_count") or 0 for d in docs_now}

docs_now = await store.snapshot(include_deprecated=True)
before = access_counts()

packaged = await retrieve(store, QUERY, top_k=TOP_K, confidence_weight=W)
assert [r["id"] for r in packaged] == [r["id"] for r in step4]

docs_now = await store.snapshot(include_deprecated=True)
after = access_counts()
returned = {r["id"] for r in packaged}
assert all(after[i] - before[i] == (1 if i in returned else 0) for i in before)
print(f"retrieve() matches the manual pipeline; access_count +1 on {len(returned)} returned memories only")

## The Payoff: The Same Question, a Different Context

`make_retrieval_tool(store)` wraps `retrieve` as an agent tool. The tool is built for one store, which is bound to one `user_id`, so the model can choose *what* to recall but never *whose* memories.

In [ ]:
recall_current_beliefs = make_retrieval_tool(store, top_k=TOP_K)

retrieval_agent = Agent(
    client=client,
    name="LifecycleRetrievalAssistant",
    instructions=SYSTEM_PROMPT + "\n\n" + (
        "Before personalizing, call recall_current_beliefs. Obey each memory's "
        "presentation exactly: assert states it as fact, hedge states it as likely "
        "and asks for confirmation, ask turns it into a question. Never use a "
        "preference the tool did not return."
    ),
    tools=[recall_current_beliefs],
)
lifecycle_answer = await retrieval_agent.run(QUESTION, session=AgentSession())
print("NAIVE:\n", naive_answer.text)
print("\n" + "=" * 70 + "\n")
print("LIFECYCLE-AWARE:\n", lifecycle_answer.text)

## Validation

The checks from the module plan, run against the live store. A second user with the same scope proves the partition boundary: the tool and `retrieve` cannot reach Sarah's memories from another `user_id`.

In [ ]:
rows = await retrieve(store, QUERY, top_k=TOP_K)
texts = [r["preference"] for r in rows]

assert "Prefers to stay at Hilton hotels" not in texts, "superseded memory leaked"
assert all(r["state"] in TRUST_STATES for r in rows), "candidate/deprecated memory leaked"
assert all(r["presentation"] in ("hedge", "ask") for r in rows if r["state"] == "provisional")
assert len(rows) <= TOP_K, "context bound exceeded"

other_user = SemanticMemoryStore(container, user_id="E002", embed_fn=embed, scope_tag=SCOPE)
assert await retrieve(other_user, QUERY) == [], "cross-user leak"

print("All retrieval checks passed")

## Key Takeaways

1. **Similarity is not eligibility.** The nearest vectors included a superseded version, a quarantined inference and retired memories.
2. **Retrieval is a policy over one query.** Partition, `current_only`, `states` and `top_k` are options on the same `search()` call, not a second storage layer.
3. **State gates visibility; confidence orders and voices.** A 0.90-confidence candidate stays hidden; among eligible memories confidence breaks ties and sets assert/hedge/ask.
4. **Bound the context.** A fixed `top_k` keeps low-value memories out without deleting anything.
5. **Track what was used.** Only returned memories update `access_count` and `last_accessed`, feeding scheduled retention.
6. **Tools are per user.** `make_retrieval_tool(store)` binds the partition key; the model never supplies it.

## Next: Memory Routing

Semantic memory is one of several surfaces. The next notebook gives one agent session, episodic, semantic, procedural and policy tools, and lets the model route each question to the right one.

In [ ]:
print(f"Removed {await store.reset()} demo memories")
await cosmos.close()